In [1]:
import numpy as np
import pandas as pd
import re
from pandas.api.types import (
    is_numeric_dtype,
    is_categorical_dtype,
    is_object_dtype,
)
from scipy.stats import spearmanr

In [2]:
train = pd.read_csv("../feature_data/all_feature_train.csv")
train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,canceled_count,applications_12m,insured_count,approval_rate,avg_application_amount,avg_credit_application_ratio,latest_credit_amount,latest_annuity,latest_days_decision,days_since_last_application
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,1.000000,179055.000,1.000000,179055.0,9251.775,-606.0,606.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,2.0,1.000000,435436.500,1.057664,1035882.0,98356.995,-746.0,746.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,1.000000,24282.000,0.828021,20106.0,5357.250,-815.0,815.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,3.0,7.0,0.0,0.555556,352265.868,0.951861,675000.0,24246.000,-181.0,181.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,3.0,1.000000,150530.250,1.046356,274288.5,16037.640,-374.0,374.0


## feature profiling

Mục tiêu không phải để quyết định giữ hay loại biến ngay lập tức mà nhằm xây dựng một bức tranh tổng thể về chất lượng của từng feature trước khi thực hiện Variable Selection.

Toàn bộ quá trình được thực hiện tự động trên tất cả các cột ngoại trừ SK_ID_CURR và TARGET. Đối với mỗi feature, cần thống kê đầy đủ các thông tin như kiểu dữ liệu, số lượng giá trị khác nhau (cardinality), tỷ lệ missing, tỷ lệ giá trị xuất hiện nhiều nhất (dominant value ratio), các thống kê mô tả đối với biến số, cũng như Bad Rate theo từng nhóm giá trị hoặc theo các quantile đối với biến liên tục.

output: một bảng Feature Profile Summary, có thể rà soát hàng trăm feature.

In [ ]:
# Feature Profile
def feature_profile(series: pd.Series):

    profile = {}

    # Basic Information
    profile["dtype"] = str(series.dtype)
    profile["missing_pct"] = series.isna().mean()
    profile["n_unique"] = series.nunique(dropna=False)

    # Dominant Value
    value_counts = series.value_counts(dropna=False)

    profile["dominant_pct"] = (
        value_counts.iloc[0] / len(series)
        if len(value_counts) > 0
        else np.nan
    )

    # Numeric Summary
    if is_numeric_dtype(series):

        profile["has_inf"] = np.isinf(series).any()

        finite = series.replace([np.inf, -np.inf], np.nan)

        profile["min"] = finite.min()
        profile["max"] = finite.max()

    else:

        profile["has_inf"] = False
        profile["min"] = np.nan
        profile["max"] = np.nan

    return profile


# Screening Rules
def detect_constant(profile):

    return profile["n_unique"] <= 1


def detect_identifier(feature_name):

    feature_name = feature_name.upper()

    keywords = [
        "SK_ID",
        "ID",
        "INDEX"
    ]

    return any(k in feature_name for k in keywords)


def detect_near_zero_variance(profile, dominant_threshold=0.90):
    return profile["dominant_pct"] >= dominant_threshold


def detect_high_missing(profile, missing_threshold=0.80):

    return profile["missing_pct"] >= missing_threshold


def detect_high_cardinality(series, profile, high_cardinality=50):
    if is_numeric_dtype(series):
        return False

    return profile["n_unique"] > high_cardinality


def detect_inf(profile):

    return profile["has_inf"]

# Decision Engine
def profile_decision(feature_name, series, profile, dominant_threshold, missing_threshold):

    # Constant
    if detect_constant(profile):

        return "Drop", "Constant Feature", "Remove"

    # Identifier
    if detect_identifier(feature_name):

        return "Drop", "Identifier", "Remove"

    # Near Zero Variance
    if detect_near_zero_variance(profile, dominant_threshold):

        return "Drop", "Near Zero Variance", "Remove"

    # Infinite Values
    if detect_inf(profile):

        return "Review", "Contains Inf", "Replace Inf Before Binning"

    # High Missing
    if detect_high_missing(profile, missing_threshold):

        return "Review", "High Missing", "Evaluate Missing Bin"

    # High Cardinality
    if detect_high_cardinality(series, profile):

        return "Review", "High Cardinality", "Consider Category Grouping"

    return "Keep", "Pass", "Fine Classing"


# Main Function
def feature_screening(df, target="TARGET", id="SK_ID_CURR", missing_threshold=0.80, dominant_threshold=0.90):

    features = [col for col in df.columns if col not in [target, id]]

    logs = []

    for feature in features:

        series = df[feature]

        profile = feature_profile(series)

        decision, reason, next_step = profile_decision(
            feature_name=feature,
            series=series,
            profile=profile,
            dominant_threshold=dominant_threshold,
            missing_threshold=missing_threshold
        )

        logs.append({

            "feature": feature,

            **profile,

            "decision": decision,

            "reason": reason,

            "next_step": next_step
        })

    profile_logs = pd.DataFrame(logs)

    keep_features = profile_logs.loc[profile_logs["decision"] != "Drop", "feature"].tolist()

    screened_train = df[[id, target] + keep_features]

    return profile_logs, screened_train

In [ ]:
profile_logs, screened_train = feature_screening(train, target = "TARGET", id = "SK_ID_CURR", missing_threshold = 0.80, dominant_threshold = 0.90)


In [5]:
profile_logs

,feature,dtype,missing_pct,n_unique,dominant_pct,has_inf,min,max,decision,reason,next_step
0,NAME_CONTRACT_TYPE,str,0.000000,2,0.904787,False,NaN,NaN,Drop,Near Zero Variance,Remove
1,CODE_GENDER,str,0.000000,3,0.658344,False,NaN,NaN,Keep,Pass,Fine Classing
2,FLAG_OWN_CAR,str,0.000000,2,0.659892,False,NaN,NaN,Keep,Pass,Fine Classing
3,FLAG_OWN_REALTY,str,0.000000,2,0.693673,False,NaN,NaN,Keep,Pass,Fine Classing
4,CNT_CHILDREN,int64,0.000000,15,0.700368,False,0.0,1.900000e+01,Keep,Pass,Fine Classing
...,...,...,...,...,...,...,...,...,...,...,...
211,avg_credit_application_ratio,float64,0.060053,236099,0.069669,False,0.2,2.740852e+00,Keep,Pass,Fine Classing
212,latest_credit_amount,float64,0.053507,49776,0.250693,False,0.0,4.085550e+06,Keep,Pass,Fine Classing
213,latest_annuity,float64,0.054863,144246,0.054863,False,0.0,3.004254e+05,Keep,Pass,Fine Classing
214,latest_days_decision,float64,0.053507,2923,0.053507,False,-2922.0,-1.000000e+00,Keep,Pass,Fine Classing


In [6]:
screened_train.head()

,SK_ID_CURR,TARGET,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,canceled_count,applications_12m,insured_count,approval_rate,avg_application_amount,avg_credit_application_ratio,latest_credit_amount,latest_annuity,latest_days_decision,days_since_last_application
0,100002,1,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,...,0.0,0.0,0.0,1.000000,179055.000,1.000000,179055.0,9251.775,-606.0,606.0
1,100003,0,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,...,0.0,0.0,2.0,1.000000,435436.500,1.057664,1035882.0,98356.995,-746.0,746.0
2,100004,0,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,...,0.0,0.0,0.0,1.000000,24282.000,0.828021,20106.0,5357.250,-815.0,815.0
3,100006,0,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,...,3.0,7.0,0.0,0.555556,352265.868,0.951861,675000.0,24246.000,-181.0,181.0
4,100007,0,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,...,0.0,0.0,3.0,1.000000,150530.250,1.046356,274288.5,16037.640,-374.0,374.0


In [7]:
screened_train.columns.tolist()

['SK_ID_CURR',
 'TARGET',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'OWN_CAR_AGE',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_PHONE',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_AVG',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',
 'LANDAREA_AVG',
 'LIVINGAPARTMENTS_AVG',
 'LIVINGAREA_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVIN

## Fine Classing

Nội dung của bước này là ứng với mỗi feature, ta chia feature đó thành nhiều nhóm (bins) nhỏ để quan sát sự thay đổi của Bad Rate trên toàn
bộ miền giá trị của biến:

- Đối với các biến liên tục, thì ta chia theo quantile nghĩa là mỗi bins sẽ chứa khoảng quan sát xấp xỉ nhau để đảm bảo Bad Rate có thể được quan sát mang tính thống kê. Với các biến phân loại thì giữ nguyên theo từng nhóm như lúc đầu (vì dạng categorical sẽ chỉ có khoảng 5-10 giá trị riêng biệt).  

- Đồng thời với missing ta xây dựng bins riêng, vì có thể nhiều biến thì việc missing cũng mang ý nghĩa tín dụng.

- Report trong đoạn code tương ứng với các chỉ số thống kê trong tùng bins của 1 feature tương ứng


Output là một bảng thống kê ứng với mỗi biến để ta quyết định xem việc merge bins ở các bước tiếp theo.  

Và sau cùng ta sẽ loại những biến có chỉ số IV (infomation Value) thuộc khoảng rất yếu

In [ ]:
ID_COL = "SK_ID_CURR"
TARGET_COL = "TARGET"

n_bins = 20
min_bin_size = 0.03 # mỗi bin phải chứa ít nhất 3% điểm dữ liệu
numeric_unique = 15 # nếu một biến số có giá trị riêng biệt <= 15 thì sẽ coi là biến phân loại
min_freq_pct = 0.01 # Gộp category có tần suất < 1%


def prepare_feature(series):
        temp = series.copy()
        if is_numeric_dtype(temp):
            temp = temp.replace([np.inf,-np.inf], np.nan)
        return temp

def create_fine_bins(series, n_bins=n_bins, numeric_unique_threshold=numeric_unique, min_freq_pct=min_freq_pct):
        if is_numeric_dtype(series):
            # Nếu số giá trị riêng biệt <= threshold, coi như categorical
            if series.nunique() <= numeric_unique_threshold:
                bins = series.fillna("missing").astype(str)
            else:
                # Fine classing với qcut
                try:
                    bins = pd.qcut(series, q=n_bins, duplicates="drop")
                    bins = bins.astype("object")
                    bins[series.isna()] = "missing"
                except Exception:
                    # Fallback: nếu qcut lỗi, xử lý như categorical
                    bins = series.fillna("missing").astype(str)
        else:
            # Xử lý biến category
            # Lấy tần suất của từng category
            freq = series.value_counts(normalize=True)
            
            # Các category có tần suất < min_freq_pct -> gộp vào "OTHER"
            other_cats = freq[freq < min_freq_pct].index.tolist()
            
            bins = series.fillna("missing").astype(str)
            bins = bins.replace(other_cats, "OTHER")
        
        return bins


def statistics(feature: pd.Series, target: pd.Series, bins: pd.Series):
        temp = pd.DataFrame({
            "feature":feature,
            "target": target,
            "bins": bins
        })
        
        table = (temp.groupby("bins").agg(
            population = ("target","count"),
            bad = ("target",  "sum")
        ).reset_index())
        
        table["good"] = table["population"] - table["bad"]
        table["bad_rate"] = table["bad"] / table['population']
        table["population_pct"] = table["population"] / table["population"].sum()

        total_bad = table["bad"].sum()
        total_good = table["good"].sum()
        
        table["pct_bad"] = table["bad"] / total_bad
        table["pct_good"] = table["good"] / total_good
        
        table["pct_bad"] = table["pct_bad"].replace(0, 0.0000001)
        table["pct_good"] = table["pct_good"].replace(0, 0.0000001)
        
        table["woe"] = np.log(table["pct_bad"] / table["pct_good"])
        
        table["iv_contribution"] = (table["pct_bad"] - table["pct_good"]) * table["woe"]
        
        return table

def numeric_summary(table: pd.DataFrame, feature: pd.Series, bins = pd.Series):
        if not is_numeric_dtype(feature):
            return table
        temp = pd.DataFrame({
            "value": feature,
            "bins": bins
        })
        
        summary = (temp.groupby("bins").agg(
            min = ("value", "min"),
            max=("value", "max"),
            mean=("value", "mean")
        ).reset_index())
        
        table = table.merge(summary, on="bins", how= "left")
        
        return table

def fine_classing(feature: pd.Series, target: pd.Series, n_bins: int = n_bins):
        feature = prepare_feature(feature)
        bins = create_fine_bins(feature, n_bins=n_bins)
        report = statistics(feature, target, bins)
        report = numeric_summary(report, feature, bins)
        
        # Tính tổng IV của biến này
        iv_total = report["iv_contribution"].sum()
        
        # Thêm cột cảnh báo 
        report["warning"] = ""
        report.loc[report["population"] < (report["population"].sum() * 0.01), "warning"] += "Small sample; "
        report.loc[report["bad"] == 0, "warning"] += "Zero bad; "
        report.loc[report["good"] == 0, "warning"] += "Zero good; "
        
        return report, iv_total


def run_fine_classing_pipeline(df: pd.DataFrame, target_col: str = TARGET_COL, id_col: str = ID_COL) -> pd.DataFrame:
        # Lấy danh sách các biến (loại trừ ID và Target)
        feature_cols = [col for col in df.columns if col not in [id_col, target_col]]
        
        results = []
        
        print(f"Bắt đầu Fine Classing cho {len(feature_cols)} biến...")
        print("=" * 60)
        
        for i, col in enumerate(feature_cols, 1):
            try:
                # Chạy fine classing cho từng biến
                report, iv = fine_classing(df[col], df[target_col])
                
                # Lưu thông tin tổng hợp
                results.append({
                    "feature": col,
                    "data_type": "numeric" if is_numeric_dtype(df[col]) else "categorical",
                    "n_unique": df[col].nunique(),
                    "n_missing": df[col].isna().sum(),
                    "missing_pct": df[col].isna().mean(),
                    "n_bins": len(report),
                    "iv": iv,
                    "iv_strength": (
                        "Very Strong" if iv > 0.5 else
                        "Strong" if iv > 0.3 else
                        "Medium" if iv > 0.1 else
                        "Weak" if iv > 0.02 else
                        "Very Weak (Drop)"
                    ),
                    "has_warning": (report["warning"] != "").any(),
                    "min_population": report["population"].min(),
                    "max_population": report["population"].max(),
                    "report": report
                })
                
                print(f"✅ [{i}/{len(feature_cols)}] {col}: IV = {iv:.4f} ({results[-1]['iv_strength']})")
                
            except Exception as e:
                print(f"❌ [{i}/{len(feature_cols)}] Lỗi với biến {col}: {str(e)[:50]}...")
                results.append({
                    "feature": col,
                    "data_type": "unknown",
                    "n_unique": np.nan,
                    "n_missing": np.nan,
                    "missing_pct": np.nan,
                    "n_bins": 0,
                    "iv": np.nan,
                    "iv_strength": "Error",
                    "has_warning": False,
                    "min_population": np.nan,
                    "max_population": np.nan,
                    "report": None
                })
        
        print("=" * 60)
        print("Hoàn thành Fine Classing!")
        
        summary_df = pd.DataFrame(results)
        
        # Sắp xếp theo IV giảm dần
        summary_df = summary_df.sort_values("iv", ascending=False).reset_index(drop=True)
        
        return summary_df

In [9]:
summary_df = run_fine_classing_pipeline(screened_train)

Bắt đầu Fine Classing cho 185 biến...
✅ [1/185] CODE_GENDER: IV = 0.0387 (Weak)
✅ [2/185] FLAG_OWN_CAR: IV = 0.0066 (Very Weak (Drop))
✅ [3/185] FLAG_OWN_REALTY: IV = 0.0005 (Very Weak (Drop))
✅ [4/185] CNT_CHILDREN: IV = 0.0072 (Very Weak (Drop))
✅ [5/185] AMT_INCOME_TOTAL: IV = 0.0118 (Very Weak (Drop))
✅ [6/185] AMT_CREDIT: IV = 0.0528 (Weak)
✅ [7/185] AMT_ANNUITY: IV = 0.0319 (Weak)
✅ [8/185] AMT_GOODS_PRICE: IV = 0.1026 (Medium)
✅ [9/185] NAME_TYPE_SUITE: IV = 0.0020 (Very Weak (Drop))
✅ [10/185] NAME_INCOME_TYPE: IV = 0.0579 (Weak)
✅ [11/185] NAME_EDUCATION_TYPE: IV = 0.0508 (Weak)
✅ [12/185] NAME_FAMILY_STATUS: IV = 0.0217 (Weak)
✅ [13/185] NAME_HOUSING_TYPE: IV = 0.0156 (Very Weak (Drop))
✅ [14/185] REGION_POPULATION_RELATIVE: IV = 0.0455 (Weak)
✅ [15/185] DAYS_BIRTH: IV = 0.0862 (Weak)
✅ [16/185] DAYS_EMPLOYED: IV = 0.1135 (Medium)
✅ [17/185] DAYS_REGISTRATION: IV = 0.0283 (Weak)
✅ [18/185] OWN_CAR_AGE: IV = 0.0234 (Weak)
✅ [19/185] FLAG_EMP_PHONE: IV = 0.0329 (Weak)
✅ [20/185

In [10]:
summary_df

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,300347,172,0.000559,21,0.623070,Very Strong,True,172,15369,bins population bad g...
1,ext_min,numeric,139146,172,0.000559,21,0.476644,Strong,True,172,15680,bins population bad ...
2,ext_max,numeric,121113,172,0.000559,21,0.451108,Strong,True,172,15766,bins population bad g...
3,EXT_SOURCE_3,numeric,814,60965,0.198253,21,0.337845,Strong,False,11381,60965,bins population bad goo...
4,EXT_SOURCE_2,numeric,119831,660,0.002146,21,0.318581,Strong,True,660,15379,bins population bad ...
...,...,...,...,...,...,...,...,...,...,...,...,...
180,WEEKDAY_APPR_PROCESS_START,categorical,7,0,0.000000,7,0.000677,Very Weak (Drop),False,16181,53901,bins population bad good bad_rat...
181,FLAG_OWN_REALTY,categorical,2,0,0.000000,2,0.000505,Very Weak (Drop),False,94199,213312,bins population bad good bad_rate p...
182,worst_pos_dpd_12m,numeric,886,100614,0.327188,2,0.000031,Very Weak (Drop),False,100614,206897,bins population bad good...
183,bad_ratio_12m,numeric,226,100614,0.327188,2,0.000031,Very Weak (Drop),False,100614,206897,bins population bad good b...


In [11]:
pass_feature_table = summary_df[summary_df["iv_strength"] != "Very Weak (Drop)"]
pass_feature_table

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,300347,172,0.000559,21,0.623070,Very Strong,True,172,15369,bins population bad g...
1,ext_min,numeric,139146,172,0.000559,21,0.476644,Strong,True,172,15680,bins population bad ...
2,ext_max,numeric,121113,172,0.000559,21,0.451108,Strong,True,172,15766,bins population bad g...
3,EXT_SOURCE_3,numeric,814,60965,0.198253,21,0.337845,Strong,False,11381,60965,bins population bad goo...
4,EXT_SOURCE_2,numeric,119831,660,0.002146,21,0.318581,Strong,True,660,15379,bins population bad ...
...,...,...,...,...,...,...,...,...,...,...,...,...
98,total_credit,numeric,205630,44020,0.143149,21,0.020879,Weak,False,12446,44020,bins population ...
99,months_observed_cc,numeric,128,220606,0.717392,21,0.020833,Weak,True,247,220606,bins population bad good ...
100,FLOORSMIN_AVG,numeric,305,208642,0.678486,10,0.020781,Weak,True,2336,208642,bins population bad good...
101,bad_months_all_x,numeric,162,44020,0.143149,4,0.020548,Weak,False,11330,240486,bins population bad good b...


## Correlation Filter

Mục tiêu của bước này để loại bỏ những biến có tương quan cao (tránh cho việc mô hình xây dựng về sau có hiện tượng đa cộng tuyến).  
Đồng thời cũng là bước để ta cố gắng giảm thiểu các biến xuống với mong muốn có thể kiểm soát đươc rõ hơn.  

Ta cũng xây dựng một ánh xạ để ứng với từng khách hàng có giá trị x thuộc bins Y ở feature A thì sẽ có điểm Woe thế nào tương ứng.  

Output sau cùng sẽ là:   
    - Bảng thống kê các chỉ số của từng biến ứng như ở bước profiling, nhưng sẽ giảm đáng kể só biến  
    - Một dataframe mới của Train nhưng được đưa về điểm WOE (woe_df)   



In [ ]:
CORRELATION_THRESHOLD = 0.7

def build_woe(report: pd.DataFrame) -> dict:
    bins_str = report["bins"].astype(str)
    woe_map = dict(zip(bins_str, report["woe"]))
    return woe_map

def transform_to_woe(feature_series: pd.Series, woe_map) -> pd.Series:
    temp = feature_series.copy()
    
    if is_numeric_dtype(temp):  
        bins = create_fine_bins(temp)  
        # Chuyển bins thành string trước khi map
        bins_str = bins.astype(str)
        woe_values = bins_str.map(woe_map)
    else:
        temp_fill = temp.fillna("missing").astype(str)
        woe_values = temp_fill.map(woe_map)
    return woe_values

def build_matrix_woe(df, pass_feature_table: pd.DataFrame) -> pd.DataFrame:
    df_woe = pd.DataFrame(index=df.index)
    
    for index, row in pass_feature_table.iterrows():
        feature_name = row["feature"]
        report = row["report"]  
        
        woe_map = build_woe(report)
        woe_values = transform_to_woe(df[feature_name], woe_map)
        df_woe[feature_name] = woe_values
    return df_woe

def corr_filter_matrix(df_woe, pass_feature_table: pd.DataFrame, threshold=CORRELATION_THRESHOLD) -> dict:
    feature = pass_feature_table["feature"].tolist()  
    iv_dict = dict(zip(pass_feature_table["feature"], pass_feature_table["iv"]))
    
    corr_matrix = df_woe[feature].corr()
    corr_mask = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
    
    high_corr_pair = []
    for i in range(len(feature)):
        for j in range(i, len(feature)):
            if corr_mask[i, j]:
                corr_value = corr_matrix.iloc[i, j]
                if abs(corr_value) > threshold:
                    high_corr_pair.append({
                        "feature_1": feature[i],
                        "feature_2": feature[j],
                        "correlation": corr_value
                    })
    
    if len(high_corr_pair) == 0:
        return {
            "final_feature_table": pass_feature_table.copy().sort_values("iv", ascending=False).reset_index(drop=True),
            "table_decision": pd.DataFrame()
        }
    
    decision = []
    feature_to_remove = set()
    for pair in high_corr_pair:
        f1 = pair["feature_1"]
        f2 = pair["feature_2"]
        corr_pair = pair["correlation"]
        iv_1 = iv_dict.get(f1, 0)
        iv_2 = iv_dict.get(f2, 0)

        if iv_1 >= iv_2:
            keep, drop = f1, f2
        else:
            keep, drop = f2, f1
            
        feature_to_remove.add(drop)
        decision.append({
            "feature_1": f1,
            "feature_2": f2, 
            "corr": corr_pair,
            "iv_1": iv_1,
            "iv_2": iv_2,
            "keep": keep,
            "drop": drop  
        })
    
    decision_df = pd.DataFrame(decision)
    
    removed_feature = list(feature_to_remove)  
    select_feature = [f for f in feature if f not in removed_feature]
    
    final_feature_table = pass_feature_table[pass_feature_table["feature"].isin(select_feature)].copy()
    final_feature_table = final_feature_table.sort_values("iv", ascending=False).reset_index(drop=True)
    
    return {
        "final_feature_table": final_feature_table,
        "table_decision": decision_df
    }

def main_corr(df, pass_feature_table):
    df_woe = build_matrix_woe(df, pass_feature_table)
    result = corr_filter_matrix(df_woe, pass_feature_table, CORRELATION_THRESHOLD)  # ✅ SỬA: Truyền đúng tham số
    
    return df_woe, result["final_feature_table"], result["table_decision"]

In [13]:
df_woe, final_feature_table, table_decision = main_corr(screened_train, pass_feature_table)
df_woe.head()

c:\Users\shina\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
C:\Users\shina\AppData\Local\Temp\ipykernel_10412\797453661.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_woe[feature_name] = woe_values
C:\Users\shina\AppData\Local\Temp\ipykernel_10412\797453661.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_woe[feature_name] = woe_values
C:\Users\shina\AppData\Local\Temp\ipykernel_10412\79745

,ext_mean,ext_min,ext_max,EXT_SOURCE_3,EXT_SOURCE_2,EXT_SOURCE_1,DAYS_EMPLOYED,employment_years,credit_utilization,AMT_GOODS_PRICE,...,approved_count,bad_ratio_all,applications_12m,FLOORSMIN_MEDI,bad_months_24m,total_credit,months_observed_cc,FLOORSMIN_AVG,bad_months_all_x,FLOORSMIN_MODE
0,1.426910,1.298241,1.278707,1.242479,0.491101,1.077213,0.347748,0.348282,NaN,0.311884,...,0.163044,-0.048299,-0.064773,0.030206,0.449055,-0.093161,-0.032085,0.002080,0.323016,0.070526
1,-0.044272,0.097349,-0.129081,0.156353,-0.319790,0.176405,0.247017,0.247300,NaN,-0.385365,...,-0.020847,-0.048299,-0.064773,-0.576319,-0.069327,-0.070068,-0.032085,-0.462372,-0.080327,-0.595555
2,-0.842433,-0.738272,-0.798119,-0.921763,-0.109046,0.058722,0.373993,0.376025,NaN,-0.238233,...,0.163044,-0.048299,-0.064773,0.072946,-0.069327,-0.043100,-0.032085,0.072946,-0.080327,0.072946
3,-0.966025,-1.066304,-0.282598,0.156353,-0.491762,0.058722,-0.120803,-0.124280,0.244025,0.162050,...,-0.141006,-0.048299,0.243939,0.072946,0.249067,0.249067,0.295035,0.072946,0.249067,0.072946
4,0.771970,-0.036108,0.925781,0.156353,0.401940,0.058722,-0.120803,-0.124280,NaN,-0.088508,...,-0.166089,-0.048299,-0.064773,0.072946,-0.069327,-0.009779,-0.032085,0.072946,-0.080327,0.072946


In [14]:
final_feature_table.head(51)

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,300347,172,0.000559,21,0.623070,Very Strong,True,172,15369,bins population bad g...
1,EXT_SOURCE_3,numeric,814,60965,0.198253,21,0.337845,Strong,False,11381,60965,bins population bad goo...
2,EXT_SOURCE_1,numeric,114584,173378,0.563811,21,0.155824,Medium,False,6705,173378,bins population ...
3,DAYS_EMPLOYED,numeric,12573,55374,0.180072,21,0.113487,Medium,False,12554,55374,bins population bad ...
4,credit_utilization,numeric,184779,45039,0.146463,17,0.103391,Medium,True,1428,77295,bins population bad good...
5,AMT_GOODS_PRICE,numeric,1002,278,0.000904,20,0.102626,Medium,True,278,35052,bins population bad ...
6,active_ratio,numeric,296,44020,0.143149,13,0.091093,Weak,True,253,52935,bins population bad good ...
7,DAYS_BIRTH,numeric,17460,0,0.000000,20,0.086246,Weak,False,15349,15397,bins population bad ...
8,credit_goods_ratio,numeric,3066,278,0.000904,15,0.084262,Weak,True,278,108470,bins population bad good ...
9,credit_history_years,numeric,802,44020,0.143149,21,0.083779,Weak,False,12074,44020,bins population bad good b...


## Manual Review

Đầu vào của Logistics Regrssion yêu cầu tính tuyến tính, vậy nên ta sẽ sử dụng kiến thức nghiệp vụ để xác định hướng của mỗi feature tương ứng (đơn điệu tăng hay giảm)  
Qua đó xây dựng hàm để ứng với kiến thức nghiệp vụ của biến A, ta muốn biến đó đơn điệu tăng hay giảm để thuận tiện cho việc giải thích model về sau.  
Đối với các biến khó xác định thì đối với quan điểm của tôi sẽ là giữ nguyên cấu trúc, vì việc ép nó là tăng hay giảm sẽ gây ra sự mất cấu trúc, dẫn tới model học sai rủi ro thực sự. Tuy nhiên việc để rõ cấu trúc có thể khiến model gặp tình trạng overfit quá mức mà không rõ nhóm rủi ro. Nên các biến này có thể được cân nhắc merge khoa học hơn ở các bước tiếp theo


Output:
- Bảng thống kê từng biến nhưng với các bins được cập nhật  
- Bảng data chỉ số thống kê cho tất cả các biến   
- Biến đổi lại woe_df ứng với report từng biến đã được cập nhật
 


In [ ]:
def merge_bins(report: pd.DataFrame, left_idx: int, right_idx: int, eps: float = 1e-6) -> pd.DataFrame:
    """
    Merge bins with support for both numeric and categorical features.
    """
    df = report.copy().reset_index(drop=True)
    
    # Convert interval bins to string if needed
    if pd.api.types.is_interval_dtype(df['bins']):
        df['bins'] = df['bins'].astype(str)
    
    # ===== KIỂM TRA: CÓ PHẢI CATEGORICAL KHÔNG? =====
    # Nếu bins không chứa dấu phẩy và không có dạng "(a, b]"
    bins_str = df['bins'].astype(str)
    is_categorical = not any(',' in str(b) and ('(' in str(b) or '[' in str(b)) for b in bins_str)
    
    # Lưu lại để dùng
    left_bin = str(df.loc[left_idx, "bins"])
    right_bin = str(df.loc[right_idx, "bins"])
    
    # Gộp dữ liệu
    df.loc[left_idx, "population"] += df.loc[right_idx, "population"]
    df.loc[left_idx, "bad"] += df.loc[right_idx, "bad"]
    df.loc[left_idx, "good"] += df.loc[right_idx, "good"]
    
    # Xử lý min/max
    left_min = df.loc[left_idx, "min"]
    right_min = df.loc[right_idx, "min"]
    df.loc[left_idx, "min"] = left_min if pd.isna(right_min) or left_min <= right_min else right_min
    
    left_max = df.loc[left_idx, "max"]
    right_max = df.loc[right_idx, "max"]
    df.loc[left_idx, "max"] = left_max if pd.isna(right_max) or left_max >= right_max else right_max
    
    #  TẠO BINS MỚI 
    if is_categorical:
        # ===== CATEGORICAL: tạo bins dạng "1+2" hoặc "A+B" =====
        # Kiểm tra nếu là số
        if left_bin.replace('.', '').replace('-', '').isdigit() and right_bin.replace('.', '').replace('-', '').isdigit():
            df.loc[left_idx, "bins"] = f"{left_bin}+{right_bin}"
        else:
            # Nếu là text, gộp thành "A+B"
            # Nếu left_bin và right_bin đã có dấu +, thì không cần thêm
            if '+' in left_bin:
                # Nếu left_bin đã có +, chỉ cần thêm right_bin
                df.loc[left_idx, "bins"] = f"{left_bin}+{right_bin}"
            else:
                df.loc[left_idx, "bins"] = f"{left_bin}+{right_bin}"
    else:
        # ===== NUMERIC: tạo interval =====
        min_val = df.loc[left_idx, "min"]
        max_val = df.loc[right_idx, "max"]
        df.loc[left_idx, "bins"] = f"({min_val}, {max_val}]"
    
    # Drop right bins
    df = df.drop(index=right_idx).reset_index(drop=True)
    
    # Statistics
    total_population = df["population"].sum()
    total_good = df["good"].sum()
    total_bad = df["bad"].sum()
    
    df["population_pct"] = df["population"] / total_population if total_population > 0 else 0
    df["bad_rate"] = np.where(df["population"] > 0, df["bad"] / df["population"], 0)
    df["pct_good"] = df["good"] / total_good if total_good > 0 else 0
    df["pct_bad"] = df["bad"] / total_bad if total_bad > 0 else 0
    
    df["woe"] = np.log((df["pct_bad"] + eps) / (df["pct_good"] + eps))
    df["iv_contribution"] = (df["pct_bad"] - df["pct_good"]) * df["woe"]
    df["mean"] = df["bad_rate"]
    
    # Warnings
    warnings = []
    for _, row in df.iterrows():
        message = []
        if row["population_pct"] < 0.03:
            message.append("Small Bin")
        if row["bad"] == 0:
            message.append("Zero Bad")
        if row["good"] == 0:
            message.append("Zero Good")
        warnings.append(", ".join(message))
    
    df["warning"] = warnings
        
    return df

    
def find_breakpoint(report, trend):
    bad_rate = report["bad_rate"].values
    break_points = []
    
    if trend == "increasing":
        for i in range(len(bad_rate) -1):
            if bad_rate[i] > bad_rate[i+1]:
                break_points.append(i)
    elif trend == "decreasing":
        for i in range(len(bad_rate) - 1):
            if bad_rate[i] < bad_rate[i+1]:
                break_points.append(i)
    else:
        raise ValueError("trend have to be inc or dcr")
    
    return break_points

    

def rank_breakpoints(report, break_points: list):
    if len(break_points) ==0:
        return None

    bad_rate = report["bad_rate"].to_numpy()
    
    n = len(report)
    
    candidates = []
    
    for bp in break_points:
        if str(report.loc[bp,"bins"]).lower() == "missing":
            continue
        if (bp +1 <n) and (str(report.loc[bp + 1,"bins"]).lower() == "missing"):
            continue
        
        if bp - 1 >= 0 and str(report.loc[bp - 1, "bins"]).lower() == "missing":
            continue
        
        
        if bp ==0:
            score = abs(bad_rate[0] - bad_rate[1])
            candidates.append({
                "left":0,
                "right": 1,
                "score": score,
                "reason": "non_monotonic"
            })
            continue
        
        if bp == n-2:
            score = abs(bad_rate[n-2] - bad_rate[n-1])
            candidates.append({
                "left":n-2,
                "right": n-1,
                "score": score,
                "reason": "non_monotonic"
            })
            continue
        
        left_score = abs(bad_rate[bp] - bad_rate[bp - 1])
        right_score = abs(bad_rate[bp] - bad_rate[bp + 1])
        
        if left_score <= right_score:
            candidates.append({

                "left":bp-1,

                "right":bp,

                "score":left_score,

                "reason":"non_monotonic"

            })
        else:
            candidates.append({

                "left":bp,

                "right":bp+1,

                "score":right_score,

                "reason":"non_monotonic"

            })
    if not candidates:
        return None
    best = min(candidates, key = lambda x: x["score"])
    return best

def main_manual(report, trend, max_iter, verbose: bool = True):
    current_report = report.copy()
    
    iteration = 1
    while iteration <= max_iter:
        if len(current_report) < 5: # số bins tối thiểu nên là 5
            break
        break_points = find_breakpoint(current_report, trend= trend)
        if len(break_points) == 0:
            if verbose:
                print("=" * 60)
                print("Finished.")
                print("No breakpoint detected.")
                print(f"Total iterations : {iteration-1}")
                print("=" * 60)

            return current_report

        merge_info = rank_breakpoints(current_report, break_points)
        if merge_info is None:
            return current_report
        
        current_report = merge_bins(current_report, left_idx=merge_info["left"], right_idx= merge_info["right"])
        
        iteration += 1
    return current_report

### Phân nhóm biên

In [16]:
# nhóm increasing
INCREASING = [
    "credit_utilization",
    "credit_goods_ratio",
    "active_ratio",
    "avg_util_6m",
    "late_payment_rate",
    "avg_credit_application_ratio",
    "active_loans",
    "REGION_RATING_CLIENT_W_CITY",
    "total_debt",
    "avg_drawings",
    "refused_count",
    "util_trend",
    "AMT_ANNUITY",
    "avg_day_late",
    "overdue_loans",
    "latest_annuity",
    "bad_months_12m_x",
    "bad_ratio_all",
    "applications_12m",
]

# nhóm decreasing
DECREASING = [
    "ext_mean",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "credit_history_years",
    "approval_rate",
    "payment_ratio_last",
    "avg_payment_12m",
    "closed_loans",
    "months_observed_all",
    "months_observed_cc",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_LAST_PHONE_CHANGE",
    "DAYS_REGISTRATION"
]

REVIEW_NUMERIC = [
    "AMT_CREDIT",
    "AMT_GOODS_PRICE",
    "total_credit",
    "avg_application_amount",
    "latest_enddate",
    "mean_future_instalments",
    "OWN_CAR_AGE",
    "ext_std",
    "FLOORSMAX_MEDI",
]

CATEGORICAL = [
    "OCCUPATION_TYPE",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "CODE_GENDER",
    "FLAG_DOCUMENT_3",
    "NAME_FAMILY_STATUS",
    "REG_CITY_NOT_WORK_CITY",
]

#features_to_drop = [
#    'payment_ratio_last',
##   'OWN_CAR_AGE',
  #  'avg_drawings',
   ## 'months_observed_cc'
#]

In [17]:
def summary_new_report(report):

    iv = report["iv_contribution"].sum()

    return {
        "n_bins": len(report),
        "iv": iv,
        "iv_strength": (
            "Very Strong" if iv > 0.5 else
            "Strong" if iv > 0.3 else
            "Medium" if iv > 0.1 else
            "Weak" if iv > 0.02 else
            "Very Weak (Drop)"
        ),
        "has_warning": (report["warning"] != "").any(),
        "min_population": report["population"].min(),
        "max_population": report["population"].max(),
    }


def update_manual_report(final_feature_table, manual_feature, trend,max_iter, verbose=False,):
    """
    Update report and feature summary after manual coarse classing.
    """

    result = final_feature_table.copy()

    manual_set = set(manual_feature)

    for idx, row in result.iterrows():

        feature = row["feature"]

        if feature not in manual_set:
            continue

        report = row["report"]

        # Merge bins
        new_report = main_manual(
            report=report,
            trend=trend,
            max_iter=max_iter,
            verbose=verbose,
        )

        summary = summary_new_report(new_report)

        # Update report
        result.at[idx, "report"] = new_report

        # Update data
        for col, value in summary.items():
            result.at[idx, col] = value
    
    return result

result = update_manual_report(final_feature_table, INCREASING, "increasing",10)
result_final_feature_table = update_manual_report(result, DECREASING, "decreasing", 10)

result_final_feature_table
    

C:\Users\shina\AppData\Local\Temp\ipykernel_10412\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval_dtype(df['bins']):
C:\Users\shina\AppData\Local\Temp\ipykernel_10412\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval_dtype(df['bins']):
C:\Users\shina\AppData\Local\Temp\ipykernel_10412\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval_dtype(df['bins']):
C:\Users\shina\AppData\Local\Temp\ipykernel_10412\1485551385.py:8: Pandas4Warning: is_interval_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.IntervalDtype)` instead
  if pd.api.types.is_interval

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,300347,172,0.000559,21,0.623070,Very Strong,True,172,15369,bins population bad g...
1,EXT_SOURCE_3,numeric,814,60965,0.198253,19,0.337808,Strong,False,11381,60965,bins ...
2,EXT_SOURCE_1,numeric,114584,173378,0.563811,19,0.155722,Medium,True,6705,173378,bins ...
3,DAYS_EMPLOYED,numeric,12573,55374,0.180072,11,0.113289,Medium,False,12554,63037,bins population bad ...
4,credit_utilization,numeric,184779,45039,0.146463,14,0.103301,Medium,False,13120,77295,bi...
5,AMT_GOODS_PRICE,numeric,1002,278,0.000904,20,0.102626,Medium,True,278,35052,bins population bad ...
6,active_ratio,numeric,296,44020,0.143149,8,0.090572,Weak,False,18781,73247,bins population bad good ...
7,DAYS_BIRTH,numeric,17460,0,0.000000,10,0.085651,Weak,False,15349,61517,bins population bad ...
8,credit_goods_ratio,numeric,3066,278,0.000904,9,0.076569,Weak,True,278,122978,bins po...
9,credit_history_years,numeric,802,44020,0.143149,11,0.082620,Weak,False,12335,65635,bins population bad good b...


In [18]:
result_final_feature_table[result_final_feature_table["feature"] == "ext_mean"]["report"].iloc[0]

,bins,population,bad,good,bad_rate,population_pct,pct_bad,pct_good,woe,iv_contribution,min,max,mean,warning
0,"(-0.00099406, 0.235]",15367,4116,11251,0.267847,0.049972,0.165801,0.039800,1.426910,1.797911e-01,0.000006,0.235432,0.161014,
1,"(0.235, 0.304]",15367,2932,12435,0.190798,0.049972,0.118107,0.043989,0.987655,7.320304e-02,0.235432,0.303868,0.272668,
2,"(0.304, 0.349]",15367,2454,12913,0.159693,0.049972,0.098852,0.045680,0.771970,4.104744e-02,0.303869,0.349495,0.327869,
3,"(0.349, 0.384]",15367,2007,13360,0.130605,0.049972,0.080846,0.047261,0.536861,1.803049e-02,0.349496,0.384269,0.367389,
4,"(0.384, 0.414]",15367,1772,13595,0.115312,0.049972,0.071380,0.048092,0.394892,9.196026e-03,0.384276,0.413648,0.399325,
5,"(0.414, 0.439]",15367,1456,13911,0.094748,0.049972,0.058651,0.049210,0.175499,1.656790e-03,0.413648,0.439198,0.426579,
6,"(0.439, 0.462]",15367,1303,14064,0.084792,0.049972,0.052487,0.049751,0.053536,1.464811e-04,0.439199,0.462289,0.450924,
7,"(0.462, 0.484]",15367,1191,14176,0.077504,0.049972,0.047976,0.050148,-0.044272,9.614381e-05,0.462289,0.484014,0.473289,
8,"(0.484, 0.505]",15367,1029,14338,0.066962,0.049972,0.041450,0.050721,-0.201840,1.871148e-03,0.484014,0.504701,0.494429,
9,"(0.505, 0.525]",15367,946,14421,0.061560,0.049972,0.038107,0.051014,-0.291713,3.765267e-03,0.504703,0.524502,0.514710,


In [19]:

def build_woe_dict(report: pd.DataFrame) -> dict:
    """Tạo dict mapping từ bins -> woe"""
    woe_dict = dict(zip(report["bins"].astype(str), report["woe"]))
    
    if "other" not in woe_dict:
        woe_dict["other"] = 0.0
    
    return woe_dict

def is_numeric_interval(interval_str: str) -> bool:
    """
    Kiểm tra xem interval có phải là numeric interval không.
    Numeric interval có dấu phẩy và dấu ngoặc: "(1, 2]"
    Categorical có dấu +: "1+2" hoặc không có dấu phẩy
    """
    if not interval_str:
        return False
    # Nếu có dấu phẩy và có dấu ngoặc → numeric interval
    if ',' in interval_str and ('(' in interval_str or '[' in interval_str):
        return True
    return False



def parse_interval(interval_str):
    """
    Parse interval string thành (left, right)
    Chỉ áp dụng cho numeric intervals
    """
    if interval_str.lower() in ['missing', 'other']:
        return None, None
    
    # Xóa các ký tự đặc biệt
    clean_str = interval_str.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
    parts = clean_str.split(",")
    
    if len(parts) != 2:
        return None, None
    
    left_str = parts[0].strip()
    right_str = parts[1].strip()
    
    # Xử lý -inf và inf
    if left_str.lower() in ["-inf", "-infinity"]:
        left = float("-inf")
    elif left_str.lower() in ["inf", "infinity"]:
        left = float("inf")
    else:
        numbers = re.findall(r"-?\d+\.?\d*", left_str)
        if not numbers:
            return None, None
        left = float(numbers[0])
    
    if right_str.lower() in ["-inf", "-infinity"]:
        right = float("-inf")
    elif right_str.lower() in ["inf", "infinity"]:
        right = float("inf")
    else:
        numbers = re.findall(r"-?\d+\.?\d*", right_str)
        if not numbers:
            return None, None
        right = float(numbers[0])
    
    return left, right

def assign_bins(feature: pd.Series, report: pd.DataFrame) -> pd.Series:
    """
    Gán bin cho feature, hỗ trợ cả numeric và categorical
    """
    bins = pd.Series(index=feature.index, dtype="object")
    
    # Xử lý missing
    bins[feature.isna()] = "missing"
    
    numeric_report = report[report["bins"] != "missing"]
    
    for _, row in numeric_report.iterrows():
        interval = str(row["bins"])
        
        # ===== XÁC ĐỊNH LOẠI BIN =====
        if is_numeric_interval(interval):
            # ===== NUMERIC INTERVAL =====
            left, right = parse_interval(interval)
            
            if left is None or right is None:
                continue
            
            # Tạo mask dựa trên left và right
            if left == float("-inf"):
                mask = (feature.notna() & (feature <= right))
            elif right == float("inf"):
                mask = (feature.notna() & (feature > left))
            else:
                mask = (feature.notna() & (feature > left) & (feature <= right))
            
            bins.loc[mask] = interval
            
        else:
            # ===== CATEGORICAL =====
            # Nếu có dấu + → gộp nhiều categories
            if '+' in interval:
                categories = [cat.strip() for cat in interval.split('+')]
                mask = (feature.notna() & (feature.astype(str).isin(categories)))
                bins.loc[mask] = interval
            else:
                # Single category
                mask = (feature.notna() & (feature.astype(str) == interval))
                bins.loc[mask] = interval
    
    # Gán "other" cho giá trị chưa được gán
    bins[bins.isna()] = "other"
    
    return bins


def transform_to_woe_manual(feature, report):
    """Chuyển đổi feature thành WOE values"""
    woe_dict = build_woe_dict(report)
    bins = assign_bins(feature, report)
    return bins.map(woe_dict)

def build_matrix_woe_manual(df, result_final_feature_table):
    """Xây dựng ma trận WOE"""
    df_woe = pd.DataFrame(index=df.index)
    
    for _, row in result_final_feature_table.iterrows():
        feature = row["feature"]
        report = row["report"]
        
        print(f"Processing feature: {feature}")
        df_woe[feature] = transform_to_woe_manual(df[feature], report)
    
    return df_woe

In [20]:
feature_result = result_final_feature_table["feature"].tolist()
last_df = screened_train.loc[:, feature_result]
df_woe_logistics = build_matrix_woe_manual(last_df, result_final_feature_table)
df_woe_logistics

Processing feature: ext_mean
Processing feature: EXT_SOURCE_3
Processing feature: EXT_SOURCE_1
Processing feature: DAYS_EMPLOYED
Processing feature: credit_utilization
Processing feature: AMT_GOODS_PRICE
Processing feature: active_ratio
Processing feature: DAYS_BIRTH
Processing feature: credit_goods_ratio
Processing feature: credit_history_years
Processing feature: OCCUPATION_TYPE
Processing feature: approval_rate
Processing feature: avg_util_6m
Processing feature: late_payment_rate
Processing feature: NAME_INCOME_TYPE
Processing feature: avg_credit_application_ratio
Processing feature: payment_ratio_last
Processing feature: latest_enddate
Processing feature: AMT_CREDIT
Processing feature: active_loans
Processing feature: REGION_RATING_CLIENT_W_CITY
Processing feature: NAME_EDUCATION_TYPE
Processing feature: total_debt
Processing feature: avg_drawings
Processing feature: refused_count
Processing feature: DAYS_LAST_PHONE_CHANGE
Processing feature: util_trend
Processing feature: avg_paym

,ext_mean,EXT_SOURCE_3,EXT_SOURCE_1,DAYS_EMPLOYED,credit_utilization,AMT_GOODS_PRICE,active_ratio,DAYS_BIRTH,credit_goods_ratio,credit_history_years,...,overdue_loans,mean_future_instalments,latest_annuity,OWN_CAR_AGE,bad_months_12m_x,NAME_FAMILY_STATUS,bad_ratio_all,applications_12m,total_credit,months_observed_cc
0,1.426910,1.242458,1.077178,0.369463,-0.141898,0.311884,-0.366184,0.365220,0.095317,0.141286,...,-0.062428,0.062812,-0.010765,0.056215,-0.071025,0.213706,-0.048299,-0.064773,-0.093161,-0.032085
1,-0.044272,0.156352,0.176398,0.261441,-0.384852,-0.385365,-0.366184,-0.029988,0.000000,-0.286466,...,-0.062428,-0.228661,-0.381572,0.056215,-0.071025,-0.071222,-0.048299,-0.064773,-0.070068,-0.032085
2,-0.842433,-0.901211,0.058722,0.369463,-0.384852,-0.238233,0.000000,-0.152263,-0.231878,-0.286466,...,-0.062428,0.054777,0.021911,0.167739,-0.071025,0.213706,-0.048299,-0.064773,-0.043100,-0.032085
3,-0.966025,0.156352,0.058722,-0.144184,0.244023,0.162050,0.249066,-0.152263,-0.231878,0.249066,...,0.249066,-0.290677,-0.067962,0.056215,0.249067,0.229088,-0.048299,0.249697,0.249067,0.351477
4,0.771970,0.156352,0.058722,-0.144184,-0.384852,-0.088508,0.000000,-0.402121,-0.231878,-0.401129,...,-0.062428,-0.084400,-0.010765,0.056215,-0.071025,0.213706,-0.048299,-0.064773,-0.009779,-0.032085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,0.394892,0.156352,1.077178,0.369463,0.244023,0.127810,0.249066,0.433255,-0.204420,0.249066,...,0.249066,0.054777,0.144795,0.056215,0.249067,0.016241,-0.048299,-0.064773,0.249067,-0.032085
307507,1.426910,0.156352,0.058722,-0.430839,0.244023,0.127810,0.249066,-0.402121,0.111288,0.249066,...,0.249066,0.054777,-0.010765,0.056215,0.249067,-0.350653,-0.048299,-0.064773,0.249067,-0.032085
307508,-0.201840,0.823518,-0.852676,-0.700259,0.117088,-0.075486,0.015981,-0.029988,0.095317,-0.401129,...,-0.062428,0.054777,0.021911,0.056215,-0.071025,0.016241,0.303284,-0.064773,-0.041209,-0.032085
307509,-0.723362,-0.657321,0.058722,-0.406103,-0.384852,0.311884,0.000000,0.213208,0.095317,-0.401129,...,-0.062428,0.062812,-0.010765,0.056215,-0.071025,-0.071222,-0.048299,0.023855,0.251536,-0.032085


In [21]:
# loại những biến sau vì tỉ lệ missing quá cao, và k có ý nghĩa
# Danh sách features cần loại bỏ
features_to_drop = [
    'payment_ratio_last',
    'util_trend', 
    'OWN_CAR_AGE',
    'avg_drawings',
    'months_observed_cc'
]

# Loại bỏ khỏi WOE matrix
df_woe_logistics_clean = df_woe_logistics.drop(columns=features_to_drop)

print(f"Original shape: {df_woe_logistics.shape}")
print(f"After dropping: {df_woe_logistics_clean.shape}")
print(f"Removed {len(features_to_drop)} features")

Original shape: (307511, 49)
After dropping: (307511, 44)
Removed 5 features


In [22]:
train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,canceled_count,applications_12m,insured_count,approval_rate,avg_application_amount,avg_credit_application_ratio,latest_credit_amount,latest_annuity,latest_days_decision,days_since_last_application
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.0,0.0,1.000000,179055.000,1.000000,179055.0,9251.775,-606.0,606.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,2.0,1.000000,435436.500,1.057664,1035882.0,98356.995,-746.0,746.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,1.000000,24282.000,0.828021,20106.0,5357.250,-815.0,815.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,3.0,7.0,0.0,0.555556,352265.868,0.951861,675000.0,24246.000,-181.0,181.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,3.0,1.000000,150530.250,1.046356,274288.5,16037.640,-374.0,374.0


In [23]:
df_woe_new = df_woe_logistics_clean.copy()
df_woe_new['SK_ID_CURR'] = train['SK_ID_CURR'].values
df_woe_new['TARGET'] = train['TARGET'].values

df_woe_new

,ext_mean,EXT_SOURCE_3,EXT_SOURCE_1,DAYS_EMPLOYED,credit_utilization,AMT_GOODS_PRICE,active_ratio,DAYS_BIRTH,credit_goods_ratio,credit_history_years,...,overdue_loans,mean_future_instalments,latest_annuity,bad_months_12m_x,NAME_FAMILY_STATUS,bad_ratio_all,applications_12m,total_credit,SK_ID_CURR,TARGET
0,1.426910,1.242458,1.077178,0.369463,-0.141898,0.311884,-0.366184,0.365220,0.095317,0.141286,...,-0.062428,0.062812,-0.010765,-0.071025,0.213706,-0.048299,-0.064773,-0.093161,100002,1
1,-0.044272,0.156352,0.176398,0.261441,-0.384852,-0.385365,-0.366184,-0.029988,0.000000,-0.286466,...,-0.062428,-0.228661,-0.381572,-0.071025,-0.071222,-0.048299,-0.064773,-0.070068,100003,0
2,-0.842433,-0.901211,0.058722,0.369463,-0.384852,-0.238233,0.000000,-0.152263,-0.231878,-0.286466,...,-0.062428,0.054777,0.021911,-0.071025,0.213706,-0.048299,-0.064773,-0.043100,100004,0
3,-0.966025,0.156352,0.058722,-0.144184,0.244023,0.162050,0.249066,-0.152263,-0.231878,0.249066,...,0.249066,-0.290677,-0.067962,0.249067,0.229088,-0.048299,0.249697,0.249067,100006,0
4,0.771970,0.156352,0.058722,-0.144184,-0.384852,-0.088508,0.000000,-0.402121,-0.231878,-0.401129,...,-0.062428,-0.084400,-0.010765,-0.071025,0.213706,-0.048299,-0.064773,-0.009779,100007,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,0.394892,0.156352,1.077178,0.369463,0.244023,0.127810,0.249066,0.433255,-0.204420,0.249066,...,0.249066,0.054777,0.144795,0.249067,0.016241,-0.048299,-0.064773,0.249067,456251,0
307507,1.426910,0.156352,0.058722,-0.430839,0.244023,0.127810,0.249066,-0.402121,0.111288,0.249066,...,0.249066,0.054777,-0.010765,0.249067,-0.350653,-0.048299,-0.064773,0.249067,456252,0
307508,-0.201840,0.823518,-0.852676,-0.700259,0.117088,-0.075486,0.015981,-0.029988,0.095317,-0.401129,...,-0.062428,0.054777,0.021911,-0.071025,0.016241,0.303284,-0.064773,-0.041209,456253,0
307509,-0.723362,-0.657321,0.058722,-0.406103,-0.384852,0.311884,0.000000,0.213208,0.095317,-0.401129,...,-0.062428,0.062812,-0.010765,-0.071025,-0.071222,-0.048299,0.023855,0.251536,456254,1


In [32]:
result_final_feature_table_last = result_final_feature_table[~result_final_feature_table["feature"].isin(features_to_drop)]

result_final_feature_table_last

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,300347,172,0.000559,21,0.623070,Very Strong,True,172,15369,bins population bad g...
1,EXT_SOURCE_3,numeric,814,60965,0.198253,19,0.337808,Strong,False,11381,60965,bins ...
2,EXT_SOURCE_1,numeric,114584,173378,0.563811,19,0.155722,Medium,True,6705,173378,bins ...
3,DAYS_EMPLOYED,numeric,12573,55374,0.180072,11,0.113289,Medium,False,12554,63037,bins population bad ...
4,credit_utilization,numeric,184779,45039,0.146463,14,0.103301,Medium,False,13120,77295,bi...
5,AMT_GOODS_PRICE,numeric,1002,278,0.000904,20,0.102626,Medium,True,278,35052,bins population bad ...
6,active_ratio,numeric,296,44020,0.143149,8,0.090572,Weak,False,18781,73247,bins population bad good ...
7,DAYS_BIRTH,numeric,17460,0,0.000000,10,0.085651,Weak,False,15349,61517,bins population bad ...
8,credit_goods_ratio,numeric,3066,278,0.000904,9,0.076569,Weak,True,278,122978,bins po...
9,credit_history_years,numeric,802,44020,0.143149,11,0.082620,Weak,False,12335,65635,bins population bad good b...


In [24]:
#df_woe_new.to_csv("woe_logistics.csv", index=False, encoding="utf-8-sig")

In [34]:
def extract_woe_bins(result_final_feature_table: pd.DataFrame) -> pd.DataFrame:
    table = []
    
    for feature_name, row in result_final_feature_table.iterrows():
        report = row["report"]
        
        temp = report.copy()
        temp["feature"] = row["feature"]
        temp["bins"] = temp["bins"].astype(str)
        
        keep_cols = ["feature", "bins", "woe", "bad_rate", "population", 
                     "population_pct", "min", "max", "mean", "warning"]
        existing_cols = [col for col in keep_cols if col in temp.columns]
        temp = temp[existing_cols]
        
        table.append(temp)
    return pd.concat(table, ignore_index= True) if table else pd.DataFrame()


In [ ]:
bins_feature = extract_woe_bins(result_final_feature_table)

#bins_feature.to_csv("bins_feature.csv", index=False,encoding="utf-8-sig")

In [33]:
result_final_feature_table

,feature,data_type,n_unique,n_missing,missing_pct,n_bins,iv,iv_strength,has_warning,min_population,max_population,report
0,ext_mean,numeric,300347,172,0.000559,21,0.623070,Very Strong,True,172,15369,bins population bad g...
1,EXT_SOURCE_3,numeric,814,60965,0.198253,19,0.337808,Strong,False,11381,60965,bins ...
2,EXT_SOURCE_1,numeric,114584,173378,0.563811,19,0.155722,Medium,True,6705,173378,bins ...
3,DAYS_EMPLOYED,numeric,12573,55374,0.180072,11,0.113289,Medium,False,12554,63037,bins population bad ...
4,credit_utilization,numeric,184779,45039,0.146463,14,0.103301,Medium,False,13120,77295,bi...
5,AMT_GOODS_PRICE,numeric,1002,278,0.000904,20,0.102626,Medium,True,278,35052,bins population bad ...
6,active_ratio,numeric,296,44020,0.143149,8,0.090572,Weak,False,18781,73247,bins population bad good ...
7,DAYS_BIRTH,numeric,17460,0,0.000000,10,0.085651,Weak,False,15349,61517,bins population bad ...
8,credit_goods_ratio,numeric,3066,278,0.000904,9,0.076569,Weak,True,278,122978,bins po...
9,credit_history_years,numeric,802,44020,0.143149,11,0.082620,Weak,False,12335,65635,bins population bad good b...
